# SMS Spam Detection — Experiment

This notebook implements the plan in [`methodology.md`](methodology.md):

1. Load and clean the SMS Spam Collection
2. Build text (TF-IDF / counts) and hand-crafted structural features
3. Train Multinomial Naive Bayes, Logistic Regression and Linear SVM
4. Evaluate on a held-out test set with spam-class F1 (primary), precision, recall, MCC and ROC-AUC

**Research question:** Can TF-IDF + a linear classifier beat a Multinomial Naive Bayes baseline on spam-class F1,
and do simple structural features add value on top of the text features?

In [1]:
import csv, re, zipfile, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (f1_score, precision_score, recall_score, accuracy_score,
                             matthews_corrcoef, roc_auc_score, ConfusionMatrixDisplay,
                             classification_report)

RANDOM_STATE = 42
pd.set_option("display.max_colwidth", 120)

## 1. Load the data

The raw file is git-ignored; download it if it is missing (see `data/README.md`).

In [2]:
DATA_DIR = Path("data/raw")
DATA_FILE = DATA_DIR / "SMSSpamCollection"
URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

if not DATA_FILE.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = DATA_DIR / "sms_spam.zip"
    urllib.request.urlretrieve(URL, zip_path)
    zipfile.ZipFile(zip_path).extractall(DATA_DIR)

# QUOTE_NONE: messages contain quote characters that would otherwise break parsing
df = pd.read_csv(DATA_FILE, sep="\t", header=None, names=["label", "message"],
                 quoting=csv.QUOTE_NONE, encoding="utf-8")
print(df.shape)
df.head()

(5574, 2)


,label,message
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std t...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives around here though"


In [3]:
print(df["label"].value_counts())
print(f"\nSpam share: {df['label'].eq('spam').mean():.1%}")

label
ham     4827
spam     747
Name: count, dtype: int64

Spam share: 13.4%


## 2. Data cleaning

In [4]:
print("Missing values:\n", df.isna().sum())
print("Empty messages:", df["message"].str.strip().eq("").sum())
print("Exact duplicate rows:", df.duplicated().sum())
print("\nDuplicates by label:\n", df[df.duplicated()]["label"].value_counts())

Missing values:
 label      0
message    0
dtype: int64
Empty messages: 0
Exact duplicate rows: 403

Duplicates by label:
 label
ham     309
spam     94
Name: count, dtype: int64


In [5]:
# Remove duplicates BEFORE splitting so the same message can't appear in both train and test
df = df.dropna().drop_duplicates().reset_index(drop=True)
df["target"] = df["label"].map({"ham": 0, "spam": 1})
print(df.shape)
print(df["label"].value_counts())
print(f"Spam share after dedup: {df['target'].mean():.1%}")

(5171, 3)
label
ham     4518
spam     653
Name: count, dtype: int64
Spam share after dedup: 12.6%


In [6]:
URL_RE = re.compile(r"(https?://\S+|www\.\S+|\S+\.(com|co\.uk|net|org)\b\S*)", re.I)
MONEY_RE = re.compile(r"[£$€]\s?\d+([.,]\d+)?|\b\d+([.,]\d+)?\s?(p|pence|pounds?)\b", re.I)
PHONE_RE = re.compile(r"\b\d{5,}\b")
NUM_RE = re.compile(r"\d+")

def clean_text(text: str) -> str:
    text = text.lower()
    text = URL_RE.sub(" urltoken ", text)
    text = MONEY_RE.sub(" moneytoken ", text)
    text = PHONE_RE.sub(" phonetoken ", text)
    text = NUM_RE.sub(" numtoken ", text)
    text = re.sub(r"[^a-z\s]", " ", text)       # drop remaining punctuation / symbols
    return re.sub(r"\s+", " ", text).strip()

df["clean"] = df["message"].apply(clean_text)
df.loc[df["target"] == 1, ["message", "clean"]].head(5)

,message,clean
2,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std t...,free entry in numtoken a wkly comp to win fa cup final tkts numtoken st may numtoken text fa to phonetoken to receiv...
5,FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX...,freemsg hey there darling it s been numtoken week s now and no word back i d like some fun you up for it still tb ok...
8,WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 0906170146...,winner as a valued network customer you have been selected to receivea moneytoken prize reward to claim call phoneto...
9,Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call Th...,had your mobile numtoken months or more u r entitled to update to the latest colour mobiles with camera for free cal...
11,"SIX chances to win CASH! From 100 to 20,000 pounds txt> CSH11 and send to 87575. Cost 150p/day, 6days, 16+ TsandCs a...",six chances to win cash from numtoken to moneytoken txt csh numtoken and send to phonetoken cost moneytoken day numt...
